# Demo 03b — Perceptron, Logistic Regression and Classification Metrics

**Class 03 · Block 2** (slides: `slides/03_linear_models.md`)

Four live demos on the **spam** data (Spambase: 4601 e-mails, 57 word/character frequencies, 39% spam):

* **C1 — The perceptron:** the update rule as a `jax.grad`, convergence, XOR.
* **C2 — Logistic regression:** the sigmoid, cross-entropy versus squared error, training with JAX, reading the weights.
* **C3 — Classification metrics:** confusion matrix, accuracy, precision, recall, $F_1$, MCC, and the threshold.
* **C4 — Polynomial features and regularization** for a curved decision boundary.

In [ ]:
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.datasets import make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../data")

In [ ]:
spam = pl.read_parquet(DATA / "spambase.parquet")
names = [c for c in spam.columns if c != "spam"]
X_raw = spam.select(names).to_numpy().astype(float)
y = spam["spam"].to_numpy().astype(int)
idx_tr, idx_te = (np.asarray(a) for a in train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0))
X_tr_raw, X_te_raw, ytr, yte = X_raw[idx_tr], X_raw[idx_te], y[idx_tr], y[idx_te]
# Most frequencies are 0 and a few are huge: log(1 + x), then standardize with the training statistics.
scaler = StandardScaler().fit(np.log1p(X_tr_raw))
Xtr, Xte = np.asarray(scaler.transform(np.log1p(X_tr_raw))), np.asarray(scaler.transform(np.log1p(X_te_raw)))
print("train", Xtr.shape, "test", Xte.shape, "| spam fraction: train", ytr.mean().round(3), "test", yte.mean().round(3))
print("duplicate e-mails in the data:", int(spam.is_duplicated().sum()), "of", spam.height, "(a leak between train and test)")

## C1 — The perceptron

$\hat y = \text{sign}(w^\top x + b)$, labels $y \in \{-1, +1\}$. The loss is the **perceptron criterion**

$$\ell(w, b) = \max\big(0,\; -y\,(w^\top x + b)\big)$$

zero for a correctly classified point, positive for a mistake. For one example with a mistake,
$\nabla_w \ell = -y\,x$ and $\nabla_b \ell = -y$, so a gradient step of length 1 is
$w \leftarrow w + y\,x,\; b \leftarrow b + y$: **the classic rule is a subgradient step**. We never write it: JAX
differentiates the loss of one example.

In [ ]:
def perceptron_loss(params, x, y_):
    margin = y_ * (x @ params["w"] + params["b"])
    return jnp.mean(jnp.where(margin <= 0, -margin, 0.0))          # a point on the boundary is a mistake


grad_fn = jax.jit(jax.grad(perceptron_loss))


def train_perceptron(X_, y_pm, epochs, record=False):
    params = {"w": jnp.zeros(X_.shape[1]), "b": jnp.zeros(())}
    X_, y_pm = jnp.asarray(X_), jnp.asarray(y_pm, dtype=float)
    mistakes, snapshots = [], []
    for _ in range(epochs):
        wrong = 0
        for i in range(len(y_pm)):
            g = grad_fn(params, X_[i : i + 1], y_pm[i : i + 1])
            wrong += int(g["b"] != 0)                              # non-zero gradient = mistake
            params = jax.tree.map(lambda p, gp: p - gp, params, g)
        mistakes.append(wrong)
        if record:
            snapshots.append((np.asarray(params["w"]).copy(), float(params["b"])))
    return params, mistakes, snapshots


def blobs(rng, n, gap):
    labels = rng.integers(0, 2, n)
    centres = np.array([[-gap / 2, -gap / 2], [gap / 2, gap / 2]])
    return centres[labels] + rng.normal(0, 1.0, (n, 2)), labels


rng = np.random.default_rng(1)
Xb, yb = blobs(rng, 60, 4.0)
_, mist, snaps = train_perceptron(Xb, 2 * yb - 1, 4, record=True)
print("separable blobs, mistakes per epoch:", mist)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, sharey=True)
xs = np.linspace(-4, 4, 10)
for ax, (w_, b_), title in zip(axes, snaps[:3], ("after epoch 1", "after epoch 2", "after epoch 3")):
    ax.scatter(*Xb[yb == 0].T, s=12, color="tab:blue")
    ax.scatter(*Xb[yb == 1].T, s=12, color="tab:red")
    ax.plot(xs, -(w_[0] * xs + b_) / w_[1], "k")
    ax.set(title=title, ylim=(-4, 4))
plt.show()

**Convergence.** If the classes are linearly separable, the perceptron makes a *finite* number of mistakes and then
stops (Novikoff's theorem: at most $(R/\gamma)^2$ updates, with $R$ the largest $\lVert x\rVert$ and $\gamma$ the margin).
If they overlap, or are not linearly separable at all, it never stops.

In [ ]:
Xo, yo = blobs(rng, 100, 1.5)
_, mist_o, _ = train_perceptron(Xo, 2 * yo - 1, 12)
print("overlapping blobs, mistakes per epoch:", mist_o)

Xx = rng.uniform(0.15, 1.0, (200, 2)) * rng.choice([-1.0, 1.0], (200, 2))            # keep a margin from the axes
yx = (Xx[:, 0] * Xx[:, 1] < 0).astype(int)                       # XOR: opposite signs
p_x, mist_x, _ = train_perceptron(Xx, 2 * yx - 1, 20)
acc_x = np.mean((np.asarray(Xx @ p_x["w"] + p_x["b"]) > 0) == yx)
print(f"XOR, raw features: mistakes per epoch {mist_x[:6]}..., accuracy {acc_x:.2f}")

P = PolynomialFeatures(2, include_bias=False).fit_transform(Xx)   # x1, x2, x1^2, x1 x2, x2^2
p_p, mist_p, _ = train_perceptron(P, 2 * yx - 1, 20)
print(f"XOR, degree-2 features: mistakes per epoch {mist_p}")

The perceptron cannot learn XOR **in the plane** (no line separates it); with the extra column $x_1 x_2$ it can. A
1969 book by Minsky and Papert made this limitation famous. We meet the way out (hidden layers) in Class 05.

**The same rule as scikit-learn.** With `eta0=1`, no penalty and no shuffling, `Perceptron` runs exactly this
online update.

In [ ]:
p_spam, mist_spam, _ = train_perceptron(Xtr, 2 * ytr - 1, 1)
sk_p = Perceptron(penalty=None, eta0=1.0, shuffle=False, tol=None, max_iter=1).fit(Xtr, ytr)  # pyright: ignore[reportArgumentType]
print("max |weights ours - sklearn| =", np.abs(np.asarray(p_spam["w"]) - np.asarray(sk_p.coef_)[0]).max(), "| mistakes in the epoch:", mist_spam[0], "of", len(ytr))
print("test accuracy after one epoch:", round(float(np.mean((np.asarray(Xte @ p_spam["w"] + p_spam["b"]) > 0) == yte)), 3))

## C2 — Logistic regression

The perceptron answers yes or no. **Logistic regression** answers with a probability:

$$p(y = 1 \mid x) = \sigma(w^\top x + b), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

$z = w^\top x + b$ is the **log-odds**: $\log\frac{p}{1-p} = z$. Training maximizes the likelihood of the labels,
which is the same as minimizing the **binary cross-entropy**

$$\ell = -\big[y \log p + (1-y)\log(1-p)\big] = \log(1 + e^{z}) - y\,z .$$

**Why not the squared error?** Compare the two losses for a positive example ($y = 1$) as a function of the logit $z$.

In [ ]:
z = np.linspace(-6, 6, 300)
sig = 1 / (1 + np.exp(-z))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(z, sig)
axes[0].set(title=r"$\sigma(z)$", xlabel="z")
axes[1].plot(z, np.log1p(np.exp(-z)), label="cross-entropy")
axes[1].plot(z, (sig - 1) ** 2, label="squared error on p")
axes[1].set(title="loss for y = 1", xlabel="z")
axes[1].legend()
axes[2].plot(z, -(1 - sig), label="d cross-entropy / dz")
axes[2].plot(z, 2 * (sig - 1) * sig * (1 - sig), label="d squared error / dz")
axes[2].set(title="gradient for y = 1", xlabel="z")
axes[2].legend()
plt.tight_layout()
plt.show()

For a confidently **wrong** prediction ($z \ll 0$) the cross-entropy gradient is $-1$ (a strong push), while the
squared-error gradient goes to 0: the model would be stuck exactly when it is most wrong. The cross-entropy is also
**convex** in $(w, b)$; the squared error on a sigmoid is not.

The gradient of the mean cross-entropy is $\frac1N X^\top(p - y)$ (a residual again), but JAX gives it to us.

In [ ]:
def bce(params, X_, y_, lam):
    z_ = X_ @ params["w"] + params["b"]
    return jnp.mean(jnp.logaddexp(0.0, z_) - y_ * z_) + 0.5 * lam * jnp.sum(params["w"] ** 2)   # log(1+e^z) - y z, + L2


bce_grad = jax.jit(jax.value_and_grad(bce))
lam = 1e-3
params = {"w": jnp.zeros(57), "b": jnp.zeros(())}
history = []
for _ in range(3000):
    value, g = bce_grad(params, Xtr, ytr.astype(float), lam)
    params = jax.tree.map(lambda p, gp: p - 0.5 * gp, params, g)
    history.append(float(value))

# scikit-learn's C multiplies the SUM of the losses: our lambda corresponds to C = 1 / (lambda * N).
sk = LogisticRegression(C=1 / (lam * len(ytr)), tol=1e-10, max_iter=10_000).fit(Xtr, ytr)
print("final training loss:", round(history[-1], 4))
print("max |weights ours - sklearn| =", float(np.abs(np.asarray(params["w"]) - sk.coef_[0]).max()))
proba = np.asarray(1 / (1 + np.exp(-(Xte @ params["w"] + params["b"]))))
print("max |test probability ours - sklearn| =", float(np.abs(proba - sk.predict_proba(Xte)[:, 1]).max()))
plt.plot(history)
plt.xlabel("gradient descent step")
plt.ylabel("training cross-entropy")
plt.show()

**Reading the weights.** The features are standardized, so $w_j$ is the change of the log-odds for one standard
deviation of feature $j$; $e^{w_j}$ is the **odds ratio**. We use the regularized version (`C = 0.1`) for stable,
interpretable weights.

In [ ]:
reg = LogisticRegression(C=0.1, max_iter=10_000).fit(Xtr, ytr)
table = pl.DataFrame({"feature": names, "weight": reg.coef_[0].round(3), "odds_ratio": np.exp(reg.coef_[0]).round(2)}).sort("weight")
print("most 'ham' features:\n", table.head(6))
print("most 'spam' features:\n", table.tail(6).reverse())

## C3 — Classification metrics

The **confusion matrix** counts the four kinds of outcome (spam is the positive class):

| | predicted spam | predicted ham |
|:--|:-:|:-:|
| **actual spam** | TP | FN |
| **actual ham** | FP | TN |

$$\text{accuracy} = \tfrac{TP+TN}{n},\quad \text{precision} = \tfrac{TP}{TP+FP},\quad \text{recall} = \tfrac{TP}{TP+FN},\quad
F_1 = \tfrac{2PR}{P+R},\quad
\text{MCC} = \tfrac{TP\,TN - FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

In [ ]:
def metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "accuracy": (tp + tn) / len(y_true),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }


proba = reg.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
m = metrics(yte, pred)
print({k: (int(v) if k in ("TP", "FP", "FN", "TN") else round(float(v), 3)) for k, v in m.items()})
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=["ham", "spam"], cmap="Blues")
plt.show()

**Accuracy hides imbalance.** Keep all the ham but only 2% spam in the test set (a mailbox with rare spam), and
compare the classifier with the lazy baseline "everything is ham".

In [ ]:
rng = np.random.default_rng(0)
ham, spam_idx = np.flatnonzero(yte == 0), np.flatnonzero(yte == 1)
keep = np.r_[ham, rng.choice(spam_idx, int(round(0.02 * len(ham) / 0.98)), replace=False)]
y_imb, p_imb = yte[keep], pred[keep]
rows = {
    "classifier": metrics(y_imb, p_imb),
    "everything is ham": {"accuracy": float(np.mean(y_imb == 0)), "precision": 0.0, "recall": 0.0, "F1": 0.0, "MCC": 0.0},
}
print(f"imbalanced test set: {len(y_imb)} e-mails, {int(y_imb.sum())} spam ({100 * y_imb.mean():.1f}%)")
for name, r in rows.items():
    print(f"  {name:<18}", {k: round(float(r[k]), 3) for k in ("accuracy", "precision", "recall", "F1", "MCC")})

Accuracy of the lazy baseline is about 98%: **the metric must match the goal**. And when a model calls everything
positive, recall is 1 and $F_1$ still looks reasonable; **MCC** (which uses all four cells) is 0. MCC is +1 for perfect
predictions, 0 for chance and $-1$ for total disagreement, whatever the class balance.

In [ ]:
all_spam = np.ones_like(yte)
print("everything is spam:", {k: round(float(v), 3) for k, v in metrics(yte, all_spam).items() if k in ("accuracy", "precision", "recall", "F1", "MCC")})

**The threshold is a decision.** The classifier returns a probability; 0.5 is just a default. A false positive
(hiding a real e-mail) costs more than a false negative (one more spam in the inbox), so we raise the threshold.

In [ ]:
ts = np.linspace(0.02, 0.98, 49)
rows = [metrics(yte, (proba >= t).astype(int)) for t in ts]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for key in ("precision", "recall", "F1", "MCC"):
    axes[0].plot(ts, [r[key] for r in rows], label=key)
axes[0].set(xlabel="threshold", ylim=(0, 1.02))
axes[0].legend(ncol=4, loc="lower center")
prec, rec, _ = precision_recall_curve(yte, proba)
axes[1].plot(rec, prec)
axes[1].set(xlabel="recall", ylabel="precision", title=f"precision-recall curve (AP {average_precision_score(yte, proba):.3f})")
plt.tight_layout()
plt.show()
print("ROC AUC (probability that a random spam scores above a random ham):", round(float(roc_auc_score(yte, proba)), 4))
for t in (0.5, 0.9, 0.99):
    r = metrics(yte, (proba >= t).astype(int))
    print(f"threshold {t}: precision {r['precision']:.3f}, recall {r['recall']:.3f}, F1 {r['F1']:.3f}, MCC {r['MCC']:.3f}, ham lost {r['FP']}")

## C4 — Polynomial features and regularization for classification

A linear classifier draws a line (a hyperplane). With polynomial features of the inputs, the *same* model draws a curve,
and with too high a degree it draws a wiggly one that memorizes the noise. Regularization ($C = 1/(\lambda N)$ in scikit-learn: small
`C` is strong regularization) keeps it smooth.

In [ ]:
Xm, ym = make_moons(200, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(2000, noise=0.3, random_state=1)
gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = np.c_[gx.ravel(), gy.ravel()]

degrees, Cs = (1, 3, 8, 15), (1e6, 0.1)
warnings.filterwarnings("ignore", category=ConvergenceWarning)          # the unregularized degree-15 fit may not converge
fig, axes = plt.subplots(2, 4, figsize=(14, 6.4), sharex=True, sharey=True)
for row, C in enumerate(Cs):
    for col, d in enumerate(degrees):
        poly = PolynomialFeatures(d, include_bias=False).fit(Xm)
        sc = StandardScaler().fit(poly.transform(Xm))
        model = LogisticRegression(C=C, max_iter=200_000, tol=1e-8).fit(sc.transform(poly.transform(Xm)), ym)
        zz = model.predict(sc.transform(poly.transform(grid))).reshape(gx.shape)
        tr_acc = model.score(sc.transform(poly.transform(Xm)), ym)
        te_acc = model.score(sc.transform(poly.transform(Xm_te)), ym_te)
        ax = axes[row, col]
        ax.contourf(gx, gy, zz, levels=[-0.5, 0.5, 1.5], colors=["#cfe3f5", "#fbdcc4"])
        ax.scatter(*Xm[ym == 0].T, s=8, color="tab:blue")
        ax.scatter(*Xm[ym == 1].T, s=8, color="tab:red")
        ax.set(title=f"degree {d}, C = {C:g}\ntrain {tr_acc:.2f}, test {te_acc:.3f}")
        print(f"degree {d:2d}, C = {C:<8g}: features {poly.n_output_features_:3d}  train acc {tr_acc:.3f}  test acc {te_acc:.3f}")
plt.tight_layout()
plt.show()

## Summary

* The perceptron's update is the subgradient of $\max(0, -m)$; it converges only on separable data.
* Logistic regression = linear score + sigmoid, trained on the cross-entropy (convex): probabilities, not just labels.
* Metrics: the confusion matrix is the source of all of them; accuracy fails on imbalance, $F_1$ ignores the true
  negatives, **MCC** uses all four cells; the decision threshold is a separate, cost-driven choice.
* Non-linear boundaries need feature expansion; regularization keeps them smooth.